OB

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)

# OOB 점수 활성화
model = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=42
)

model.fit(X, y)

print("OOB Score:", model.oob_score_)





OOB Score: 0.9533333333333334
OOB Score: 0.96
CV Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
CV Mean Score: 0.9666666666666668


In [3]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 데이터 불러오기
X, y = load_iris(return_X_y=True)

# 랜덤 포레스트 (OOB 사용)
rf_oob = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,   # OOB 평가 활성화
    random_state=42,
    n_jobs=-1
)
rf_oob.fit(X, y)

# OOB 점수 출력
print("OOB Score:", rf_oob.oob_score_)

# 교차검증 (CV, 5-폴드)
cv_scores = cross_val_score(rf_oob, X, y, cv=5, n_jobs=-1)
print("CV Scores:", cv_scores)
print("CV Mean Score:", cv_scores.mean())

OOB Score: 0.96
CV Scores: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
CV Mean Score: 0.9666666666666668


In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# === 1. 데이터 불러오기 ===
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# === 2. 랜덤포레스트 모델 정의 ===
rf = RandomForestClassifier(random_state=42)

# === 3. 하이퍼파라미터 그리드 정의 ===
param_grid = {
    "n_estimators": [50, 100, 200],   # 트리 개수
    "max_depth": [None, 5, 10],       # 트리 최대 깊이
    "min_samples_split": [2, 5, 10],  # 노드 분할 최소 샘플 수
    "min_samples_leaf": [1, 2, 4],    # 리프 최소 샘플 수
    "max_features": ["sqrt", "log2"]  # 특성 선택 방식
}

# === 4. GridSearchCV 실행 ===
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,                # 5-겹 교차검증
    n_jobs=-1,           # 병렬처리
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

# === 5. 결과 출력 ===
print("최적 하이퍼파라미터:", grid_search.best_params_)
print("최적 교차검증 정확도:", grid_search.best_score_)

# === 6. 최적 모델 평가 ===
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
print("테스트 정확도:", accuracy_score(y_test, y_pred))


최적 하이퍼파라미터: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
최적 교차검증 정확도: 0.9428571428571428
테스트 정확도: 1.0


In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# === 1. 데이터 불러오기 ===
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# === 2. 랜덤포레스트 모델 정의 ===
rf = RandomForestClassifier(random_state=42)

# === 3. 탐색할 하이퍼파라미터 분포 정의 ===
param_dist = {
    "n_estimators": np.arange(50, 301, 50),   # 50~300 사이 트리 개수
    "max_depth": [None, 5, 10, 15, 20],       # 트리 깊이
    "min_samples_split": np.arange(2, 11),    # 분할 최소 샘플 수
    "min_samples_leaf": np.arange(1, 6),      # 리프 최소 샘플 수
    "max_features": ["sqrt", "log2", None]    # 특성 선택 방식
}

# === 4. RandomizedSearchCV 실행 ===
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,            # 무작위 샘플링 횟수
    cv=5,                 # 5-겹 교차검증
    n_jobs=-1,            # 병렬 처리
    random_state=42,
    scoring="accuracy"
)

random_search.fit(X_train, y_train)

# === 5. 결과 출력 ===
print("최적 하이퍼파라미터:", random_search.best_params_)
print("최적 교차검증 정확도:", random_search.best_score_)

# === 6. 최적 모델 평가 ===
best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
print("테스트 정확도:", accuracy_score(y_test, y_pred))


최적 하이퍼파라미터: {'n_estimators': np.int64(150), 'min_samples_split': np.int64(10), 'min_samples_leaf': np.int64(1), 'max_features': 'sqrt', 'max_depth': 5}
최적 교차검증 정확도: 0.9428571428571428
테스트 정확도: 1.0


In [8]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from skopt import BayesSearchCV

# 데이터 불러오기 (iris 데이터셋)
X, y = load_iris(return_X_y=True)

# 모델 정의
model = RandomForestClassifier(random_state=42)

# 탐색할 하이퍼파라미터 범위 지정
param_space = {
    'n_estimators': (10, 200),          # 트리 개수
    'max_depth': (1, 20),               # 트리 깊이
    'min_samples_split': (2, 20),       # 내부 노드 분할 최소 샘플 수
    'min_samples_leaf': (1, 20)         # 리프 노드 최소 샘플 수
}

# BayesSearchCV 설정
opt = BayesSearchCV(
    estimator=model,
    search_spaces=param_space,
    n_iter=32,                # 총 32번 탐색
    cv=3,                     # 3-fold 교차검증
    random_state=42,
    n_jobs=-1                 # 병렬 처리
)

# 최적화 실행
opt.fit(X, y)

# 결과 출력
print("최적 하이퍼파라미터:", opt.best_params_)
print("최적 성능 (CV score):", opt.best_score_)

최적 하이퍼파라미터: OrderedDict({'max_depth': 11, 'min_samples_leaf': 4, 'min_samples_split': 6, 'n_estimators': 83})
최적 성능 (CV score): 0.9733333333333333


In [7]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.9 MB/s eta 0:00:00


In [9]:
import time
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# === 1. 데이터 불러오기 ===
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# === 2. 랜덤포레스트 모델 ===
rf = RandomForestClassifier(random_state=42)

# === 3. 하이퍼파라미터 그리드 (GridSearch 용) ===
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# === 4. 하이퍼파라미터 분포 (RandomSearch 용) ===
param_dist = {
    "n_estimators": np.arange(50, 301, 10),
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": np.arange(2, 21),
    "min_samples_leaf": np.arange(1, 11),
    "max_features": ["sqrt", "log2", None]
}

# === 5. GridSearchCV 실행 ===
start = time.time()
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring="accuracy")
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

# === 6. RandomizedSearchCV 실행 ===
start = time.time()
random_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=30, cv=5, n_jobs=-1, scoring="accuracy", random_state=42
)
random_search.fit(X_train, y_train)
random_time = time.time() - start

# === 7. 결과 출력 ===
print("GridSearch 최적 파라미터:", grid_search.best_params_)
print("GridSearch 교차검증 정확도:", grid_search.best_score_)
print("GridSearch 수행 시간: %.2f 초" % grid_time)

print("\nRandomizedSearch 최적 파라미터:", random_search.best_params_)
print("RandomizedSearch 교차검증 정확도:", random_search.best_score_)
print("RandomizedSearch 수행 시간: %.2f 초" % random_time)

# === 8. 테스트 세트 평가 ===
best_model_grid = grid_search.best_estimator_
best_model_random = random_search.best_estimator_

y_pred_grid = best_model_grid.predict(X_test)
y_pred_random = best_model_random.predict(X_test)

print("\n테스트 정확도 (GridSearch):", accuracy_score(y_test, y_pred_grid))
print("테스트 정확도 (RandomizedSearch):", accuracy_score(y_test, y_pred_random))


GridSearch 최적 파라미터: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
GridSearch 교차검증 정확도: 0.9428571428571428
GridSearch 수행 시간: 91.35 초

RandomizedSearch 최적 파라미터: {'n_estimators': np.int64(160), 'min_samples_split': np.int64(3), 'min_samples_leaf': np.int64(7), 'max_features': 'log2', 'max_depth': 20}
RandomizedSearch 교차검증 정확도: 0.9523809523809523
RandomizedSearch 수행 시간: 23.79 초

테스트 정확도 (GridSearch): 1.0
테스트 정확도 (RandomizedSearch): 1.0


In [ ]:
import time
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# BayesSearchCV 불러오기
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# === 1. 데이터 불러오기 ===
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# === 2. 랜덤포레스트 모델 ===
rf = RandomForestClassifier(random_state=42)

# === 3. GridSearch 파라미터 ===
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# === 4. RandomizedSearch 파라미터 분포 ===
param_dist = {
    "n_estimators": np.arange(50, 301, 10),
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": np.arange(2, 21),
    "min_samples_leaf": np.arange(1, 11),
    "max_features": ["sqrt", "log2", None]
}

# === 5. BayesianSearch 파라미터 공간 ===
param_space = {
    "n_estimators": Integer(50, 300),
    "max_depth": Integer(2, 20),
    "min_samples_split": Integer(2, 20),
    "min_samples_leaf": Integer(1, 10),
    "max_features": Categorical(["sqrt", "log2", None])
}

# === 6. GridSearch ===
start = time.time()
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring="accuracy")
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

# === 7. RandomizedSearch ===
start = time.time()
random_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=30, cv=5, n_jobs=-1, scoring="accuracy", random_state=42
)
random_search.fit(X_train, y_train)
random_time = time.time() - start

# === 8. BayesianSearch ===
start = time.time()
bayes_search = BayesSearchCV(
    rf, search_spaces=param_space,
    n_iter=30, cv=5, n_jobs=-1, scoring="accuracy", random_state=42
)
bayes_search.fit(X_train, y_train)
bayes_time = time.time() - start

# === 9. 결과 출력 ===
print("GridSearch 최적 파라미터:", grid_search.best_params_)
print("GridSearch 교차검증 정확도:", grid_search.best_score_)
print("GridSearch 수행 시간: %.2f 초" % grid_time)

print("\nRandomizedSearch 최적 파라미터:", random_search.best_params_)
print("RandomizedSearch 교차검증 정확도:", random_search.best_score_)
print("RandomizedSearch 수행 시간: %.2f 초" % random_time)

print("\nBayesianSearch 최적 파라미터:", bayes_search.best_params_)
print("BayesianSearch 교차검증 정확도:", bayes_search.best_score_)
print("BayesianSearch 수행 시간: %.2f 초" % bayes_time)

# === 10. 테스트 세트 평가 ===
best_models = {
    "GridSearch": grid_search.best_estimator_,
    "RandomizedSearch": random_search.best_estimator_,
    "BayesianSearch": bayes_search.best_estimator_
}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    print(f"\n테스트 정확도 ({name}): {accuracy_score(y_test, y_pred):.4f}")
